# 03 · Analysis and figures

Hours 24-30. Concept-level bootstrap (10,000 resamples), the specificity index,
the exchange rate, and the four figures. Runs on CPU -- no GPU needed.

In [ ]:
#@title Clone the repo and install dependencies { display-mode: "form" }
# Colab: paste a GitHub PAT with repo:read scope. It is used only for the clone
# and is not written to disk.
import os, subprocess, sys, getpass, pathlib

REPO   = "sagnikc395/apart-mind-digital-mind"  #@param {type:"string"}
BRANCH = "main"                                 #@param {type:"string"}
WORKDIR = "/content"

if pathlib.Path("/content").exists():
    token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token (blank if public): ")
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    dest = pathlib.Path(WORKDIR) / REPO.split("/")[-1]
    if dest.exists():
        subprocess.run(["git", "-C", str(dest), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, str(dest)], check=True)
    os.chdir(dest)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "torch", "transformers>=4.44", "accelerate", "datasets", "matplotlib"], check=True)
else:
    os.chdir(pathlib.Path.cwd())  # already inside the repo, e.g. running locally

sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
print("cwd:", os.getcwd())


In [ ]:
#@title Persist results to Drive (survives a session kill)
import os, pathlib

RESULTS = "/content/drive/MyDrive/alignment_tax_results"  #@param {type:"string"}
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    RESULTS = str(pathlib.Path.cwd() / "results")
    print("no Drive; writing to", RESULTS, f"({exc})")
os.environ["ALIGNMENT_TAX_RESULTS"] = RESULTS
pathlib.Path(RESULTS).mkdir(parents=True, exist_ok=True)
print("results ->", RESULTS)


In [ ]:
from pathlib import Path
import os, json
from alignment_tax.config import RunConfig
from alignment_tax import pipeline

results = Path(os.environ.get("ALIGNMENT_TAX_RESULTS", "results"))
cfg = RunConfig.load(next(results.rglob("run_config.json")))
cfg.results_dir = results

analysis = pipeline.stage_analyse(cfg, n_boot=10_000)
print(json.dumps(analysis["exchange_rate"], indent=2))

In [ ]:
import pandas as pd
pd.DataFrame(analysis["rows"])[[
    "lam", "tpr", "fpr_clean", "fpr_random", "identification",
    "conditional_identification", "d_clean", "d_random", "specificity_index",
    "safety_refusal_rate", "cap_mmlu", "cap_truthfulqa_mc1", "cap_ce_loss",
]].round(3)

In [ ]:
#@title Figures 1-4
paths = pipeline.stage_figures(cfg)
from IPython.display import Image, display
for p in paths:
    display(Image(str(p)))

## Judge validation

Hand-label 100 identification outputs and report Cohen's kappa against the
grader. Run `judge-sample`, fill in the `human` field, then `judge-kappa`.

In [ ]:
!python -m alignment_tax.cli judge-sample --results-dir $ALIGNMENT_TAX_RESULTS
# ... fill in the 'human' field in judge_labels.jsonl, then:
!python -m alignment_tax.cli judge-kappa --results-dir $ALIGNMENT_TAX_RESULTS

In [ ]:
#@title Statistical contrasts (two-proportion tests, Holm-corrected)
print(json.dumps(analysis["contrasts"]["detection_C1"], indent=2))
print(json.dumps(analysis["forced_choice_vs_chance"], indent=2))